# Phase 5: Fine-tune a small model with QLoRA (Google Colab)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. In Google Drive, create `MyDrive/clinical-note-qa/` and upload `train.parquet`, `val.parquet` and `eval_400.parquet` from your local `data/` folder.
3. Click the key icon (**Secrets**) in the left sidebar, add `HF_TOKEN` with your write token, and turn on notebook access.

Run top to bottom. For the first pass keep `N_TRAIN = 2000`; once it works end to end, raise it to 15,000–20,000 and rerun from step 3.

## 1. Set up Colab

In [ ]:
!pip install -q -U transformers peft trl accelerate bitsandbytes datasets

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/clinical-note-qa"
login(token=userdata.get("HF_TOKEN"))

HF_USER = "peytonramsey"
ADAPTER_REPO = f"{HF_USER}/qwen-clinical-note-qa-lora"
N_TRAIN = 2000   # start small; raise to 15_000-20_000 once this works
N_VAL = 300
OUT_DIR = f"{DATA}/checkpoints-{N_TRAIN}"

## 2. Load the base model in 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)  # T4 has no bf16
# dtype=float16 keeps the frozen, non-quantized layers out of bf16 (Qwen's default)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                             dtype=torch.float16, device_map="auto")

## 3. Format the data as chat messages

Same system prompt as Phase 3. The data is split into a `prompt` (system + note + question) and a `completion` (the answer), so the loss is computed only on the answer tokens. With a single `messages` list, as in the guide, the model would also be trained to reproduce the note, which wastes most of the training signal.

In [ ]:
from datasets import load_dataset

SYSTEM = ("You answer questions about a clinical discharge summary. "
          "Use only information in the note. If the note does not contain "
          "the answer, say so. Be concise.")

def prompt_messages(note, question):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Note:\n{note}\n\nQuestion: {question}"}]

def to_example(r):
    return {"prompt": prompt_messages(r["note"], r["question"]),
            "completion": [{"role": "assistant", "content": r["answer"]}]}

train = load_dataset("parquet", data_files=f"{DATA}/train.parquet", split="train")
val = load_dataset("parquet", data_files=f"{DATA}/val.parquet", split="train")
train = train.shuffle(seed=42).select(range(N_TRAIN))
val = val.shuffle(seed=42).select(range(N_VAL))
train = train.map(to_example, remove_columns=train.column_names)
val = val.map(to_example, remove_columns=val.column_names)
train[0]

## 4. Configure LoRA and train

Checkpoints go to Drive, so a Colab disconnect doesn't lose your work. If Colab disconnects, rerun steps 1–3 and this cell: it resumes from the latest checkpoint automatically.

Watch the logs: training loss should fall, and validation loss should fall then flatten. If validation loss starts rising, you're overfitting, so stop and use the best checkpoint (`load_best_model_at_end` picks it for you).

In [ ]:
import glob
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules="all-linear", task_type="CAUSAL_LM")

cfg = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,      # effective batch size 16
    learning_rate=2e-4,
    max_length=2048,                    # notes are at most ~870 words, so nothing is cut off
    completion_only_loss=True,          # train on the answer only
    gradient_checkpointing=True,
    fp16=True, bf16=False,               # T4 has no bf16; newer versions may turn it on by default
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = SFTTrainer(model=model, args=cfg, train_dataset=train,
                     eval_dataset=val, peft_config=lora, processing_class=tok)

# TRL casts LoRA weights to bf16 for 4-bit models, but the T4 has no bf16 and fp16 training
# can't unscale bf16 gradients. Keep the trainable weights in fp32 (the optimizer is built in train()).
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

resume = bool(glob.glob(f"{OUT_DIR}/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume)

In [ ]:
import pandas as pd
log = pd.DataFrame(trainer.state.log_history)
log[["step", "loss", "eval_loss"]].groupby("step").first().plot(marker=".", title="loss")

## 6. Save the adapter to Hugging Face
(Step 5, scaling up, is just raising `N_TRAIN` in step 1 and rerunning.)

In [ ]:
trainer.model.push_to_hub(ADAPTER_REPO, commit_message=f"LoRA on {N_TRAIN} train rows")
tok.push_to_hub(ADAPTER_REPO)
print(f"https://huggingface.co/{ADAPTER_REPO}")

## 7. Predict on the 400 evaluation questions

Runs **both** the untuned base model and the fine-tuned model, so you can show the fine-tuning helped. Greedy decoding (the Hugging Face equivalent of `temperature=0`), one question at a time so latency is comparable with Phase 3.

This cell loads the adapter from Hugging Face, so it also works in a fresh session after a disconnect (run steps 1–2 first). Results save to Drive every 25 rows and resume if rerun. Each model takes roughly 30–60 minutes on a T4.

In [ ]:
import os, time
import pandas as pd
from peft import PeftModel

if "trainer" in globals():
    ft_model = trainer.model
else:
    ft_model = PeftModel.from_pretrained(model, ADAPTER_REPO)
ft_model.eval()

eval_set = pd.read_parquet(f"{DATA}/eval_400.parquet")
os.makedirs(f"{DATA}/results", exist_ok=True)

@torch.no_grad()
def ask(m, note, question):
    enc = tok.apply_chat_template(prompt_messages(note, question), add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(m.device)
    n_in = enc["input_ids"].shape[1]
    start = time.perf_counter()
    out = m.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
    latency = time.perf_counter() - start
    new = out[0, n_in:]
    return {"prediction": tok.decode(new, skip_special_tokens=True).strip(),
            "input_tokens": n_in, "output_tokens": len(new), "latency_s": latency}

def predict(m, strategy, path):
    done = set(pd.read_csv(path).row_id) if os.path.exists(path) else set()
    buffer = []
    for row_id, r in eval_set.iterrows():
        if row_id in done:
            continue
        buffer.append({"row_id": row_id, "task": r.task, "question": r.question, "answer": r.answer,
                       "strategy": strategy, "model": BASE if strategy == "base" else ADAPTER_REPO,
                       **ask(m, r.note, r.question)})
        if len(buffer) >= 25:
            pd.DataFrame(buffer).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
            buffer.clear()
            print(strategy, "saved through", row_id)
    if buffer:
        pd.DataFrame(buffer).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    return pd.read_csv(path)

finetuned = predict(ft_model, "finetuned", f"{DATA}/results/finetuned.csv")
with ft_model.disable_adapter():   # same weights, adapter switched off = the untuned base model
    base_small = predict(ft_model, "base", f"{DATA}/results/base_small.csv")

### Quick comparison
Token F1 is the same rough overlap score used in Phase 4; the real evaluation is Phase 7.

In [ ]:
import re
from collections import Counter

def token_f1(pred, ref):
    p, r = re.findall(r"[a-z0-9]+", str(pred).lower()), re.findall(r"[a-z0-9]+", str(ref).lower())
    common = sum((Counter(p) & Counter(r)).values())
    if not common:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

res = pd.concat([finetuned, base_small])
res["token_f1"] = [token_f1(p, a) for p, a in zip(res.prediction, res.answer)]
print(len(finetuned), len(base_small))
display(res.groupby("strategy")[["input_tokens", "output_tokens", "latency_s", "token_f1"]].mean().round(3))
display(res.pivot_table(index="task", columns="strategy", values="token_f1").round(3))

**Checkpoint:** the adapter is on your Hugging Face account, and `results/finetuned.csv` and `results/base_small.csv` (in Drive) each have 400 rows. Download both into the local `results/` folder for Phases 6–7.